In [1]:
import pandas as pd
import mysql.connector
import os

In [ ]:
# List of CSV files and their corresponding table names
csv_files = [
    ('customers.csv', 'customers'),
    ('orders.csv', 'orders'),
    ('sellers.csv', 'sellers'),
    ('products.csv', 'products'),
    ('geolocation.csv', 'geolocation'),
    ('payments.csv', 'payments'),
    ('order_items.csv', 'order_items')
]

# Connect to the MySQL database
conn = mysql.connector.connect(
    host='localhost',
    user='root',
    password='password',
    database='commerce'
)
cursor = conn.cursor()

# Folder containing the CSV files
folder_path = r"E:\SQL prjects\Target Sales project\Target Sales Dataset"

def get_sql_type(dtype):
    if pd.api.types.is_integer_dtype(dtype):
        return 'INT'
    elif pd.api.types.is_float_dtype(dtype):
        return 'FLOAT'
    elif pd.api.types.is_bool_dtype(dtype):
        return 'BOOLEAN'
    elif pd.api.types.is_datetime64_any_dtype(dtype):
        return 'DATETIME'
    else:
        return 'TEXT'

BATCH_SIZE = 5000  # 🔑 prevents max_allowed_packet error

for csv_file, table_name in csv_files:
    file_path = os.path.join(folder_path, csv_file)

    print(f"\n🚀 Processing {csv_file}")

    # Read CSV
    df = pd.read_csv(file_path)

    # Replace NaN with None (for SQL NULL)
    df = df.where(pd.notnull(df), None)

    # Clean column names
    df.columns = [
        col.replace(' ', '_').replace('-', '_').replace('.', '_')
        for col in df.columns
    ]

    # Create table
    columns_def = ', '.join(
        f"`{col}` {get_sql_type(df[col].dtype)}"
        for col in df.columns
    )
    create_table_query = f"""
    CREATE TABLE IF NOT EXISTS `{table_name}` (
        {columns_def}
    )
    """
    cursor.execute(create_table_query)

    # Build INSERT query once
    columns = ', '.join(f"`{col}`" for col in df.columns)
    placeholders = ', '.join(['%s'] * len(df.columns))
    insert_query = f"""
    INSERT INTO `{table_name}` ({columns})
    VALUES ({placeholders})
    """

    # Prepare data
    values = [
        tuple(None if pd.isna(x) else x for x in row)
        for row in df.values
    ]

    # 🔥 CHUNKED INSERT (FIX)
    for i in range(0, len(values), BATCH_SIZE):
        batch = values[i:i + BATCH_SIZE]
        cursor.executemany(insert_query, batch)
        conn.commit()

        print(f"   → Inserted {i + len(batch)} rows...")

    print(f"✅ Finished `{table_name}`")

# Close connection
cursor.close()
conn.close()

print("\n🎉 All CSV files imported successfully!")



🚀 Processing customers.csv
   → Inserted 5000 rows...
   → Inserted 10000 rows...
   → Inserted 15000 rows...
   → Inserted 20000 rows...
   → Inserted 25000 rows...
   → Inserted 30000 rows...
   → Inserted 35000 rows...
   → Inserted 40000 rows...
   → Inserted 45000 rows...
   → Inserted 50000 rows...
   → Inserted 55000 rows...
   → Inserted 60000 rows...
   → Inserted 65000 rows...
   → Inserted 70000 rows...
   → Inserted 75000 rows...
   → Inserted 80000 rows...
   → Inserted 85000 rows...
   → Inserted 90000 rows...
   → Inserted 95000 rows...
   → Inserted 99441 rows...
✅ Finished `customers`

🚀 Processing orders.csv
   → Inserted 5000 rows...
   → Inserted 10000 rows...
   → Inserted 15000 rows...
   → Inserted 20000 rows...
   → Inserted 25000 rows...
   → Inserted 30000 rows...
   → Inserted 35000 rows...
   → Inserted 40000 rows...
   → Inserted 45000 rows...
   → Inserted 50000 rows...
   → Inserted 55000 rows...
   → Inserted 60000 rows...
   → Inserted 65000 rows...
  